# Cell Type Visualization per Tissue

Standalone notebook: for each tissue, render a multi-panel page (one panel per
cell type present) of the H&E image overlaid with that cell type's segmented
boundaries, and save everything to a single multi-page PDF.

Designed for on-screen zoom/inspection (large pages), not printing.

In [ ]:
from pathlib import Path
import math
import gc

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.backends.backend_pdf import PdfPages

import spatialdata as spd
import spatialdata_plot  # noqa: F401  # registers the .pl accessor on zdata

/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


## Locations

In [2]:
project_folder = Path("/home/janzules/spatial/CAR-T/")

# Save all cell-type visualizations here
figures_folder = project_folder / "figures"
figures_folder.mkdir(parents=True, exist_ok=True)

# Data file (SpatialData zarr store)
zarr_file = "/coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/processing_Zarr/1_C2l_annotations_400"

## Load Data

In [3]:
zdata = spd.read_zarr(zarr_file)
adata = zdata["segmentation_counts"]
adata

AnnData object with n_obs × n_vars = 2088557 × 19059
    obs: 'sample', 'cell_id', 'region', 'TMA', 'mouse', 'tissue', 'condition', 'tumor_loc', 'replicate_num', 'c2l_argmax', 'c2l_permissive', 'c2l_strict', 'c2l_runnerup', 'c2l_consolidated', 'treatment'
    uns: 'normalization', 'spatialdata_attrs', 'log1p'
    obsm: 'c2l_q95', 'c2l_q05', 'spatial', 'c2l_means'
    layers: 'counts'

## Render each tissue in its own process

Memory leaks inside `spatialdata-plot` / `dask` / `matplotlib` accumulate across
hundreds of panel renders and eventually OOM a single long-lived kernel. To avoid
that, each tissue is rendered by a **separate short-lived subprocess**
(`render_tissue.py`): it draws that tissue's page, writes a PNG, and exits, so the
OS reclaims all of its memory (leak included) before the next tissue starts.

The per-tissue PNGs are then concatenated into one PDF, one page at a time, so the
final assembly stays light too. The run is **resumable**: already-rendered PNGs in
`figures/_per_tissue/` are skipped, so if it dies partway you can just re-run.

In [ ]:
import sys
import gc
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.backends.backend_pdf import PdfPages

# ---------------------------------------------------------------------------
# This notebook is edited locally but the *kernel runs on the remote server*,
# so a standalone .py file on the laptop is not visible to the kernel. To stay
# self-contained, the per-tissue worker is embedded here and written to the
# server filesystem (under figures_folder) at runtime. Subprocesses then run it
# with the same interpreter as this kernel (sys.executable).
# ---------------------------------------------------------------------------

WORKER_SOURCE = r'''
#!/usr/bin/env python
"""Render ONE tissue's cell-type page to a PNG, then exit.

Run as a short-lived subprocess (one per tissue) so that any memory leaked by
spatialdata-plot / dask / matplotlib during rendering is reclaimed by the OS
when the process exits. The driver notebook calls this once per tissue and then
concatenates the per-tissue PNGs into a single PDF.

Exit codes:
    0  success (PNG written)
    2  skipped (missing image/shape key, or no non-Unknown cell types)
    1  error (traceback printed)
"""
from __future__ import annotations

import argparse
import math
import sys
import warnings

import matplotlib
matplotlib.use("Agg")  # headless; no display needed in the worker

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

import spatialdata as spd
import spatialdata_plot  # noqa: F401  # registers the .pl accessor


# Fixed cell-type display order (lineage-grouped), excluding 'Unknown'.
CELL_TYPE_ORDER = [
    # Non-immune / structural
    "Cancer_cell",
    "Erythrocyte",
    "Endothelial",
    "Fibroblast",
    # Monocytes
    "Monocyte",
    "Classical_Mono",
    "Nonclassical_Mono",
    # Macrophages
    "Macrophage",
    "M1_like_Mac",
    "M2_like_Mac",
    "Intermediate_Mac",
    # Neutrophils
    "Neutrophil",
    "N1_like_Neu",
    "N2_like_Neu",
    # Dendritic cells
    "DC",
    "cDC",
    "pDC",
    # T cells
    "Tcell",
    "CD4_T",
    "CD8_T",
    "Treg",
    # NK / NKT
    "NK",
    "NKT",
    # B cells
    "B",
]

# Treatment escalation order: control -> chemo+nonspecific CAR-T ->
# chemo+specific CAR-T -> +radiation(nonspecific) -> +radiation(specific).
TREATMENT_RANK = {
    "NoTx": 0,
    "CyT72": 1,
    "CyPSCA": 2,
    "RTCyT72": 3,
    "RTCyPSCA": 4,
}

# Single high-contrast color used for every panel.
PANEL_COLOR = "#0033CC"  # strong blue


def tissue_sort_key(tissue):
    """Sort tissues by treatment -> tumor location -> replicate.

    Name convention: [treatment]_[tumor_location]_[replicate]
    e.g. 'RTCyPSCA_1_2' = RT+Cy+PSCA, directly-irradiated tumor (1), replicate 2.
    Tumor location 1 = directly irradiated; 2 = contralateral (abscopal).
    """
    parts = str(tissue).split("_")
    treatment = parts[0]
    tumor_loc = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0
    replicate = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else 0
    return (TREATMENT_RANK.get(treatment, 99), tumor_loc, replicate)


def render_tissue(
    zarr_file,
    tissue,
    out_png,
    label_col="c2l_label_perm",
    tissue_col="tissue",
    table_name="segmentation_counts",
    dpi_sel=200,
    ncols=4,
    panel_size=6.0,
    image_scale=None,
):
    zdata = spd.read_zarr(zarr_file)

    img_key = f"{tissue}_hires_tissue_image"
    shape_key = f"{tissue}_cell_boundaries"

    if img_key not in zdata.images:
        print(f"Skipping {tissue}: missing image key {img_key}")
        return 2
    if shape_key not in zdata.shapes:
        print(f"Skipping {tissue}: missing shape key {shape_key}")
        return 2

    obs = zdata.tables[table_name].obs
    tissue_mask = (obs[tissue_col].astype(str) == tissue).to_numpy()
    labels_here = obs.loc[tissue_mask, label_col]

    present_cell_types = [
        ct for ct in CELL_TYPE_ORDER if (labels_here == ct).any()
    ]

    if len(present_cell_types) == 0:
        print(f"Skipping {tissue}: no non-Unknown cell types found")
        return 2

    # Subset to just this tissue's elements so render_shapes joins ~tens of
    # thousands of rows, not the whole 2M-cell table.
    sdata_t = zdata.subset([img_key, shape_key], filter_tables=True)

    n_panels = len(present_cell_types)
    nrows = math.ceil(n_panels / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * panel_size, nrows * panel_size),
        dpi=dpi_sel,
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).ravel()
    fig.suptitle(
        f"{tissue}\nCell types present: {len(present_cell_types)}",
        fontsize=22,
    )

    for ax, cell_type in zip(axes, present_cell_types):
        print(f"  {tissue}: {cell_type}")
        (
            sdata_t.pl
            .render_images(
                img_key,
                scale=image_scale,
                norm=Normalize(0, 255),
                alpha=0.85,
            )
            .pl.render_shapes(
                shape_key,
                color=label_col,
                groups=[cell_type],
                palette=[PANEL_COLOR],
                table_name=table_name,
                fill_alpha=1,
                outline_alpha=0.95,
                outline_width=0.35,
                method="matplotlib",
            )
            .pl.show(
                ax=ax,
                coordinate_systems=["downscale_to_hires"],
                dpi=dpi_sel,
                colorbar=False,
            )
        )
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()
        for leg in list(fig.legends):
            leg.remove()
        ax.set_title(cell_type, fontsize=14)
        ax.set_axis_off()

    for ax in axes[len(present_cell_types):]:
        ax.set_axis_off()

    fig.savefig(out_png, dpi=dpi_sel)
    plt.close(fig)
    print(f"Wrote {out_png} ({n_panels} cell types)")
    return 0


def main():
    p = argparse.ArgumentParser(description="Render one tissue's cell-type page.")
    p.add_argument("--zarr", required=True)
    p.add_argument("--tissue", required=True)
    p.add_argument("--out", required=True, help="output PNG path")
    p.add_argument("--label-col", default="c2l_label_perm")
    p.add_argument("--tissue-col", default="tissue")
    p.add_argument("--table-name", default="segmentation_counts")
    p.add_argument("--dpi", type=int, default=200)
    p.add_argument("--ncols", type=int, default=4)
    p.add_argument("--panel-size", type=float, default=6.0)
    p.add_argument("--image-scale", default="None",
                   help="multiscale level name, or 'None' for auto")
    args = p.parse_args()

    image_scale = None if args.image_scale == "None" else args.image_scale

    warnings.filterwarnings("ignore", category=FutureWarning)
    warnings.filterwarnings("ignore", category=UserWarning)

    try:
        code = render_tissue(
            zarr_file=args.zarr,
            tissue=args.tissue,
            out_png=args.out,
            label_col=args.label_col,
            tissue_col=args.tissue_col,
            table_name=args.table_name,
            dpi_sel=args.dpi,
            ncols=args.ncols,
            panel_size=args.panel_size,
            image_scale=image_scale,
        )
    except Exception:
        import traceback
        traceback.print_exc()
        return 1
    return code


if __name__ == "__main__":
    sys.exit(main())
'''

worker_dir = Path(figures_folder) / "_render_worker"
worker_dir.mkdir(parents=True, exist_ok=True)
worker_py = worker_dir / "render_tissue.py"
worker_py.write_text(WORKER_SOURCE)
print(f"Worker written to: {worker_py}")


# Treatment escalation order: control -> chemo+nonspecific CAR-T ->
# chemo+specific CAR-T -> +radiation(nonspecific) -> +radiation(specific).
TREATMENT_RANK = {
    "NoTx": 0,
    "CyT72": 1,
    "CyPSCA": 2,
    "RTCyT72": 3,
    "RTCyPSCA": 4,
}


def tissue_sort_key(tissue):
    """Sort tissues by treatment -> tumor location -> replicate."""
    parts = str(tissue).split("_")
    treatment = parts[0]
    tumor_loc = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0
    replicate = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else 0
    return (TREATMENT_RANK.get(treatment, 99), tumor_loc, replicate)


def render_all_tissues_isolated(
    zarr_file,
    tissues,
    figures_folder,
    final_pdf_name="c2l_label_perm_all_tissues_celltypes_present.pdf",
    label_col="c2l_label_perm",
    table_name="segmentation_counts",
    dpi_sel=200,
    ncols=4,
    panel_size=6.0,
    image_scale=None,
):
    """Render one tissue per subprocess, then concatenate PNGs into one PDF."""
    figures_folder = Path(figures_folder)
    png_dir = figures_folder / "_per_tissue"
    png_dir.mkdir(parents=True, exist_ok=True)

    tissues_sorted = sorted([str(t) for t in tissues], key=tissue_sort_key)
    n = len(tissues_sorted)
    png_paths = []

    for i, tissue in enumerate(tissues_sorted):
        out_png = png_dir / f"{i:02d}_{tissue}.png"

        # Resumable: skip tissues already rendered.
        if out_png.exists():
            print(f"[{i + 1}/{n}] {tissue}: already rendered, skipping")
            png_paths.append(out_png)
            continue

        print(f"[{i + 1}/{n}] {tissue}: rendering in subprocess ...")
        cmd = [
            sys.executable, str(worker_py),
            "--zarr", str(zarr_file),
            "--tissue", tissue,
            "--out", str(out_png),
            "--label-col", label_col,
            "--table-name", table_name,
            "--dpi", str(dpi_sel),
            "--ncols", str(ncols),
            "--panel-size", str(panel_size),
            "--image-scale", "None" if image_scale is None else str(image_scale),
        ]
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.stdout:
            print(proc.stdout.rstrip())

        if proc.returncode == 0 and out_png.exists():
            png_paths.append(out_png)
        elif proc.returncode == 2:
            print(f"   -> skipped ({tissue})")
        else:
            print(f"   -> ERROR rendering {tissue} (exit {proc.returncode})")
            if proc.stderr:
                print(proc.stderr.rstrip())

    # Concatenate PNGs into one PDF, one page at a time (bounded memory).
    final_pdf = figures_folder / final_pdf_name
    with PdfPages(final_pdf) as pdf:
        for p in png_paths:
            arr = mpimg.imread(p)
            h, w = arr.shape[0], arr.shape[1]
            fig = plt.figure(figsize=(w / dpi_sel, h / dpi_sel), dpi=dpi_sel)
            ax = fig.add_axes([0, 0, 1, 1])
            ax.imshow(arr)
            ax.set_axis_off()
            pdf.savefig(fig, dpi=dpi_sel)
            plt.close(fig)
            del arr
            gc.collect()

    print(f"\nDone. {len(png_paths)} pages.\nFinal PDF: {final_pdf}")
    return final_pdf

Worker written to: /home/janzules/spatial/CAR-T/figures/_render_worker/render_tissue.py


## Generate the PDF

Tissues are ordered by experimental structure (treatment escalation -> tumor
location -> replicate) inside `render_all_tissues_isolated`.

To test on a single tissue first, pass e.g. `tissues=["NoTx_1_1"]`.

In [ ]:
tissues = (
    zdata.tables["segmentation_counts"].obs["tissue"].astype(str).unique()
)

final_pdf = render_all_tissues_isolated(
    zarr_file=zarr_file,
    tissues=tissues,
    figures_folder=figures_folder,
    label_col="c2l_label_perm",
    dpi_sel=200,       # ~1200 px/panel at panel_size=6
    ncols=4,           # fixed grid -> consistent, large pages
    panel_size=6.0,    # inches per panel; raise for bigger zoomable pages
    image_scale=None,  # None = auto-pick pyramid level to match panel
)

print(f"Saved PDF to:\n{final_pdf}")

[1/32] NoTx_1_1: rendering in subprocess ...


: 

: 

: 